## **Lo que van a encontrar en este notebook**

Tenemos tres metas:

1. Ver la filtracion de Rips en ejemplos sencillos y calcular sus codigos de barras.

2. Repasar el algoritmo de SW1PerS (Sliding Windows and 1-Persistence Scoring) para cuantificar periodicidad/recurrencia en series de tiempo.

3. Utilizar el algoritmo SW1PerS para ordenar por periodicidad algunas series de tiempo en un conjunto sintetico.

Completar las actividades marcadas con **Hacer** o **Tu Respuesta**. Sugiero leer atentamente el codigo y los comentarios para tener una idea de que esta pasando.


## Parte I: La Filtracion de Rips y sus Numeros de Betti

Para este ejemplo generamos una nube de puntos en el espacio Euclideo.

In [ ]:
import numpy as np

# plotting and visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Numerical
from scipy.spatial import distance

np.random.seed(1223)

n_data = 30
theta = np.random.uniform(0, 2*np.pi, n_data)
data = np.array([np.cos(theta) , np.sin(theta) , np.zeros_like(theta)]).T
data += np.random.normal(0, 0.08, data.shape)

fig = go.Figure(data=[go.Scatter3d(
    x=data.T[0], y=data.T[1], z=data.T[2],
    mode ='markers',
    marker=dict(size = 3 , color = 'grey'))])

fig.update_layout(scene= dict(zaxis = dict(range=[-1, 1])))
fig.show()

### Filtracion de Rips

Dado un espacio metrico $(X,\mathbf{d}_X)$, su filtracion de Rips es:

$$\mathcal{R}(X) = \Big\{\,  R_\alpha(X,\mathbf{d}_X) \Big\}_{ \alpha \geq 0} $$
donde
$$R_\alpha(X, \mathbf{d}_X) = \Big\{\, \{x_0 , \ldots, x_k\} \subseteq X  \;\; : \;\;  \max_{0 \leq i \leq j \leq k} \mathbf{d}_X(x_i, x_j) < \alpha\, \Big\} $$

A continuacion vamos a calcular el complejo de Rips de los datos en le circulo para varios valores de $\alpha$

In [ ]:
distMat = distance.squareform(distance.pdist(data))

z_min, z_max = -1, 1

def build_complex(alpha):
    """Vectorized edge + triangle construction for a given alpha."""
    adj = distMat < alpha
    np.fill_diagonal(adj, False)

    # ---- Edges (fully vectorized) ----
    i_idx, j_idx = np.where(np.triu(adj, k=1))

    nan_col = np.full(i_idx.shape[0], np.nan)
    e_x = np.stack([data[i_idx, 0], data[j_idx, 0], nan_col], axis=1).ravel()
    e_y = np.stack([data[i_idx, 1], data[j_idx, 1], nan_col], axis=1).ravel()
    e_z = np.stack([data[i_idx, 2], data[j_idx, 2], nan_col], axis=1).ravel()

    # ---- Triangles (loop only over existing edges, not all i,j pairs) ----
    ii, jj, kk = [], [], []
    for i, j in zip(i_idx, j_idx):
        candidates = np.where(adj[i] & adj[j])[0]
        candidates = candidates[candidates > j]
        for k in candidates:
            ii.append(i)
            jj.append(j)
            kk.append(k)

    return e_x, e_y, e_z, ii, jj, kk


def make_traces(alpha):
    """Return the (vertices, edges, triangles) traces for one alpha value."""
    e_x, e_y, e_z, ii, jj, kk = build_complex(alpha)

    vertices = go.Scatter3d(
        mode="markers", name="vertices",
        x=data[:, 0], y=data[:, 1], z=data[:, 2],
        marker=dict(size=3, color="grey"),
    )
    edges = go.Scatter3d(
        mode="lines", name="aristas",
        x=e_x, y=e_y, z=e_z,
        line=dict(color="rgb(70,70,70)", width=1),
    )
    triangles = go.Mesh3d(
        x=data[:, 0], y=data[:, 1], z=data[:, 2],
        i=ii, j=jj, k=kk,
        color="lightpink", opacity=0.2,
    )
    return vertices, edges, triangles


# ---------------------------------------------------------------
# Interactive slider
# ---------------------------------------------------------------
alpha_values = np.arange(0.0, 2.01, 0.05)

base_vertices, base_edges, base_triangles = make_traces(alpha_values[0])

frames = [
    go.Frame(
        data=list(make_traces(a)),
        name=f"{a:.2f}",
        layout=go.Layout(title=f"Rips complex (alpha = {a:.2f})"))
    for a in alpha_values]

fig = go.Figure(
    data=[base_vertices, base_edges, base_triangles],
    frames=frames)

fig.update_traces(hoverinfo="none")
fig.update_layout( 
    title=f"Vietoris–Rips complex (alpha = {alpha_values[0]:.2f})",
    width=900, height=800,
    scene=dict(
        xaxis=dict(showspikes=False),
        yaxis=dict(showspikes=False),
        zaxis=dict(showspikes=False, range=[z_min, z_max]),
    ),
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "alpha = "},
        "pad": {"t": 50},
        "steps": [
            {
                "args": [[f.name], {"frame": {"duration": 0, "redraw": True},
                                     "mode": "immediate"}],
                "label": f.name,
                "method": "animate",
            }
            for f in frames
        ],
    }],
)

fig.show()


**Hacer:** Use la celda anterior para visualizar el complejo de Rips de $X$ para varios valores de $\alpha \geq 0 $. Completar los valores faltantes:

1.   $\beta_0(R_\alpha(X, \mathbf{d}_X)) = 5$   cuando  $\alpha = 0.40$

2.   $\beta_1(R_\alpha(X, \mathbf{d}_X)) = 1$   cuando  $\alpha = 1.00$

3.   $\beta_0(R_\alpha(X, \mathbf{d}_X)) = 1$ al mismo tiempo que  $\beta_1(R_\alpha(X, \mathbf{d}_X)) = 0 $
 cuando $\alpha = 0.70$

**Tu respuesta (detalle):** los valores anteriores no son unicos, sino representantes de intervalos entre eventos de la filtracion. Leyendo los codigos de barra de la Parte II (que confirman lo que se ve en el slider):

* $\beta_0 = 5$ para todo $\alpha \in (0.348,\; 0.540)$ &nbsp; — &nbsp; ahi caben $\alpha = 0.40,\, 0.45,\, 0.50$.
* $\beta_1 = 1$ para todo $\alpha \in (0.775,\; 1.685)$ &nbsp; — &nbsp; la unica clase de $H_1$ nace en $\alpha \approx 0.78$ (cuando el anillo de puntos se conecta en un ciclo) y muere en $\alpha \approx 1.69$ (cuando los triangulos rellenan el disco).
* $\beta_0 = 1$ y $\beta_1 = 0$ simultaneamente en dos regimenes: la ventana corta $\alpha \in (0.657,\; 0.775)$, donde ya todo esta conectado pero el ciclo aun no se ha cerrado, y $\alpha > 1.685$, donde el complejo ya es contractil. En el slider, $\alpha = 0.70$ es el valor mas comodo.

Moraleja: solo el intervalo largo $[0.78,\, 1.69)$ de $\beta_1$ es *robusto*; los demas dependen mucho del valor exacto de $\alpha$. Eso es justo lo que mide la persistencia.

---

## Parte II: Calculando de Codigos de Barra (barcodes) via Ripser

U. Bauer: "Ripser is a lean C++ code for the computation of Vietoris–Rips persistence barcodes. It can do just this one thing, but does it extremely well."

Libreria C++  : https://github.com/Ripser/ripser

Libreria de Python library: https://ripser.scikit-tda.org/en/latest

Articulo : https://arxiv.org/pdf/1908.02518.pdf

In [ ]:
!pip install ripser

A continuacion calculamos la persistencia de la filtracion de Rips de los datos usando `Ripser`

In [ ]:
import matplotlib.pyplot as plt

# topological data analysis
from ripser import ripser
from persim import plot_diagrams


def plot_barcodes(diagrams, alpha_max, width = 1.5):
    max_dim = len(diagrams)
    fig, axs = plt.subplots(max_dim)
    fig.suptitle('Barcodes')
    for dim in range(max_dim):
        barcode = np.copy(diagrams[dim])
        ind_inf = np.isinf(barcode.T[1])
        barcode[ind_inf, 1] = alpha_max
        h = 1
        for i in range(len(barcode)):
            x = barcode[i]
            y = [h,h]
            axs[dim].plot(x, y, linestyle= '-', c='#1f77b4', linewidth = width)
            if ind_inf[i]:
                axs[dim].scatter([alpha_max],[h],  s=10, marker='>', c='#1f77b4')
            h += 1
        axs[dim].set_xlim(0, 1.05*alpha_max)
        axs[dim].set_ylim(0,h)
        axs[dim].get_yaxis().set_ticks([]);
        axs[dim].spines['right'].set_color('none')
        axs[dim].spines['top'].set_color('none')
        axs[dim].text(0.3, 1, r'$\mathrm{bcd}^{\mathcal{R}}_{'+str(dim)+'}(X)$', verticalalignment='bottom')


# Persistence Computation
rips_persistence = ripser(data, maxdim=1)

dgms = rips_persistence['dgms']
plot_barcodes(dgms,1.8);

### Diagramas de persistencia

Los diagramas de persistencia son una visualizacion alternativa de los codigos de barra, que puede ser util para datos mas grandes:

$$ \mathrm{dgm}_j^\mathcal{R}(X) = \Big\{ (a,b) \in \mathbb{R}^2 \;\; : \;\; [a,b) \in \mathrm{bcd}_j^\mathcal{R}(X)  \Big\}$$

In [ ]:
plt.figure(figsize = (3,3))
plot_diagrams(dgms, title='Persistence Diagrams')

---

## Ejemplo: Toro ruidoso

In [ ]:
np.random.seed(2)
n_data = 25000
R = 5
r = 2
data = np.zeros((3, n_data))
s = np.random.rand(n_data)*2*np.pi
t = np.random.rand(n_data)*2*np.pi

data[0] = (R + r*np.cos(s))*np.cos(t)
data[1] = (R + r*np.cos(s))*np.sin(t)
data[2] = r*np.sin(s)
data += 0.1*np.random.randn(*data.shape)
data = data.T

# Plot the data
fig = go.Figure(data=[go.Scatter3d(
    x=data.T[0], y=data.T[1], z=data.T[2],
    mode ='markers',
    marker=dict(size = 1.5 , color = 'grey'))])

fig.update_layout( width=900, height=500)
fig.show()

In [ ]:
## Persistence Computation

# Parameters
max_alpha = 5.35
max_homology_dim = 2

# Run Ripser
rips_persistence = ripser(data,  maxdim=max_homology_dim , thresh = max_alpha , n_perm =200)

# Visualize barcodes
dgms = rips_persistence['dgms']
plot_barcodes(dgms,max_alpha);

In [ ]:
# Plot persistence diagrams
plt.figure(figsize = (3,3))
plot_diagrams(dgms)

**Preguntas**

1.  Sea $T$ el toro y $X$  el conjunto de datos arriba.
Encuentre  $\alpha$ tal que $\beta_j(R_\alpha(X)) = \beta_j(T)$ para $0 \leq j \leq 2$.

2. Que visualizacion fue mas util en esta tarea, los barcodes o los diagramas de persistencia?

**Tu respuesta:**

1. $\alpha = 3.0$

   El toro tiene $\beta_0(T)=1$, $\beta_1(T)=2$, $\beta_2(T)=1$. Contando barras vivas en cada $\alpha$ se obtiene el perfil correcto $(1,2,1)$ para todo $\alpha \in (2.27,\; 3.49)$, asi que $\alpha = 3.0$ esta comodamente en el centro de esa ventana. Los eventos que la delimitan son:
   * $\alpha \approx 1.71$: muere la ultima componente conexa espuria $\Rightarrow \beta_0 = 1$.
   * $\alpha \approx 2.26$: nace la clase de $H_2$ (la cavidad del toro) y ya solo quedan vivas las dos barras largas de $H_1$ $\Rightarrow (1,2,1)$.
   * $\alpha \approx 3.49$: muere el ciclo pequeno de $H_1$ (el que da la vuelta por el "tubo", de radio $r=2$) $\Rightarrow \beta_1$ baja a 1.
   * $\alpha \approx 3.92$: muere la cavidad de $H_2$; el ciclo grande de $H_1$ (el que rodea el agujero central, radio $R=5$) sobrevive hasta $\alpha \approx 5.30$.

   Notar que las dos barras largas de $H_1$ tienen persistencias muy distintas ($\approx 3.82$ y $\approx 2.16$) porque los dos ciclos generadores tienen radios distintos ($R=5$ y $r=2$): la escala geometrica se lee directamente en el codigo de barras.

2. Los **codigos de barra**. Con `n_perm = 200` hay cientos de clases de ruido y en el diagrama de persistencia todas se apinan sobre la diagonal, encimadas unas con otras y mezcladas entre dimensiones; ademas, para responder la pregunta 1 hace falta contar cuantas clases estan *vivas* a un $\alpha$ dado, y en el barcode eso es literalmente trazar una recta vertical en $\alpha$ y contar cuantas barras corta — en el diagrama habria que contar puntos en el cuadrante $\{a < \alpha \leq b\}$, que es mucho menos inmediato. El diagrama gana cuando lo que interesa es solo *cuanto* persiste cada clase (distancia a la diagonal) o cuando hay tantas clases que las barras ya no caben en la figura, y es el objeto adecuado para calcular distancias (cuello de botella, Wasserstein) y vectorizaciones.

---

## Parte III: Ventanas Deslizantes (Sliding Windows)

Dada una serie de tiempo $f: I \subseteq \mathbb{R} \rightarrow  \mathbb{R}$ un parametro de dilacion $\tau > 0$ y una dimension  $d+1 \in \mathbb{N}$, las **ventanas deslizantes** (sliding windows) de $f$ en $t \in I$ es

$$SW_{d,\tau}f(t) = \begin{bmatrix}f(t) \\ f(t + \tau)  \\ \vdots \\ f(t + d\tau)  \end{bmatrix}  \in \mathbb{R}^{d+1}$$

y la **nube de ventanas deslizantes** (sliding window point cloud) es

$$\mathbb{SW}_{d,\tau} f = \{SW_{d,\tau}(t) \;\; : \;\; t\in I\} \subseteq \mathbb{R}^{d+1}$$

In [ ]:
from scipy.interpolate import CubicSpline

def SW_cloud(f, tau, d, n_data):
    # Inputs:
    # f : time series -- array of size (2,N) (x and y values) or (1,N) (only y values)
    # tau: delay -- positive real number
    # d : gives embedding dimension d+1 -- integer
    # n_data : desired number of points in SW point cloud -- intenger
    #
    # Output:
    # SW : sliding window point cloud -- array of size (n_data,  d+1)

    #Step 1: turn f into a cubic spline
    if len(f.shape)==1:
        N = len(f)
        x_vals = np.linspace(0,1,N)
        y_vals = f
    else:
        x_vals = f[0]
        y_vals = f[1]

    f = CubicSpline(x_vals , y_vals)

    #Step 2: create the t values where to evaluate SW_f
    t_vals = np.linspace(np.min(x_vals) , np.max(x_vals) - d*tau, n_data)

    #Step 3: evaluate the sliding window point cloud
    SW = []
    for t in t_vals:
        SW_f_t = f(t + np.arange(0,d+1)*tau)
        SW.append(SW_f_t)

    return np.array(SW)

### Ejemplo:

Una serie de tiempo periodica con un poco de ruido

In [ ]:
# Time Series Example : Noisy sin(t)

period = 2*np.pi   ## <-- sin(t) is periodic with period 2pi
t_vals = np.linspace(0 , 5*period, 1000)
noise_level = 0.1

y_vals  = np.sin(t_vals) + noise_level*np.random.randn(*t_vals.shape)

plt.figure(figsize = (9,2))
plt.plot(t_vals, y_vals);
plt.title('Serie de tiempo');
plt.xlabel('$t$');
plt.ylabel('$f(t)$');


a continuacion calculamos la nube de ventanas deslizantes

In [ ]:
## Compute the Sliding window point cloud

f = np.array([t_vals, y_vals]) ## <---- serie de tiempo

# Parametros para SW
d = 2
tau = period/(d+1)  ## <--- la teoria de SW dice que estos son los mejores parametros

n_data = 5000

# Calcular SW
SW_f = SW_cloud(f,tau, d, n_data )

# Visualizar la nube de ventanas
fig = go.Figure(data=[go.Scatter3d(
    x=SW_f.T[0], y=SW_f.T[1], z=SW_f.T[2],
    mode ='markers',
    marker=dict(size = 1.5 , color = 'grey'))])

fig.update_layout( width=900, height=450)
fig.show()

**Preguntas**
1. Que le pasa a la nube de ventanas deslizantes si se aumenta el ruido de la serie de tiempo? Experimentar con el codigo arriba.


2. Que le pasa a $\mathbb{SW}_{d,\tau} f$  si $\tau$ cambia (e.g., a 0.1 o 6.1). Experimentar con el codigo arriba.


**Tu respuesta:**

1. Con `noise_level = 0` la nube es una **curva cerrada de grosor cero**: una elipse (topologicamente un circulo) contenida en un plano de $\mathbb{R}^3$. Al subir el ruido esa curva se *engorda*: en lugar de un circulo se ve un anillo/toroide difuso alrededor de la misma elipse, y el grosor del anillo crece proporcionalmente al nivel de ruido. Mientras el grosor sea menor que el radio de la elipse, el agujero central sigue siendo visible (y la barra larga de $H_1$ sobrevive); cuando el ruido se vuelve comparable a la amplitud de la senal (aqui $\gtrsim 0.5$) el agujero se tapa y la nube se ve como una bola solida. Ojo con un efecto secundario: el ruido tambien *agranda* el diametro de la nube, asi que la persistencia cruda en unidades absolutas puede subir aunque la forma sea peor — por eso en la Parte IV normalizamos.

2. $\tau$ controla la "redondez" de la elipse. La nube siempre esta en la curva $t \mapsto (\sin t, \sin(t+\tau), \sin(t+2\tau))$, pero su forma cambia mucho:
   * $\tau = 0.1$ (muy chico): $f(t) \approx f(t+\tau) \approx f(t+2\tau)$, asi que todos los puntos caen casi sobre la **diagonal** $x = y = z$. La elipse esta aplastada a un segmento y practicamente no hay agujero (persistencia maxima en $H_1 \approx 0.08$).
   * $\tau = 6.1 \approx 2\pi$ (un periodo completo): otra vez $f(t+\tau) \approx f(t)$, con lo cual la nube vuelve a colapsar sobre la diagonal (persistencia $\approx 0.15$). Lo mismo pasa con cualquier $\tau$ cercano a un multiplo del periodo.
   * $\tau = 2\pi/3 = $ `period/(d+1)`: la ventana $d\tau$ cubre $2/3$ del periodo y los tres retardos quedan repartidos uniformemente en el ciclo; la elipse es **maximamente redonda** (un circulo en el plano perpendicular a la diagonal) y el agujero es lo mas grande posible. Esa es exactamente la razon de la formula $\tau = \text{periodo}/(d+1)$.

   En resumen: hay periodicidad en la senal para *cualquier* $\tau$, pero solo con $\tau$ cerca del optimo la geometria de $\mathbb{SW}_{d,\tau}f$ la hace *detectable* por la homologia persistente.

---
La siguiente celda calcula la persistencia homologica de la nube de ventanas deslizantes

In [ ]:
## Persistencia homologica de SW_f

# Serie de tiempo
period = 2*np.pi   ## <-- sin(t) is periodic with period 2pi
noise_level = 0.1

t_vals = np.linspace(0 , 5*period, 1000)
y_vals  = np.sin(t_vals) + noise_level*np.random.randn(*t_vals.shape)
f = np.array([t_vals, y_vals]) ## <---- serie de tiempo

# Parametros para SW
d = 2
tau = period/(d+1)  ## <--- la teoria de SW dice que estos son los mejores parametros

# Calcular SW
SW_f = SW_cloud(f,tau, d, n_data )

# Parametros para ripser/ persistencia
max_alpha = 1.8
max_homology_dim = 1

# Correr Ripser
rips_persistence = ripser(SW_f,  maxdim=max_homology_dim , thresh = max_alpha , n_perm =200)

# Visualizar barcodes
dgms = rips_persistence['dgms']
plot_barcodes(dgms,max_alpha);

La longitud de la barra mas larga (su persistencia) en $\mathsf{bcd}^\mathcal{R}_1(X)$, para $X= \mathbb{SW}_{d,\tau}f$, esta calculada abajo

In [ ]:
# Persistencia 1-dimensional maxima
max_pers_dim_1 = np.max(dgms[1][:, 1] - dgms[1][:, 0])
print('La persistencia maxima en bcd_1 es', max_pers_dim_1)

**Preguntas:**

1. Si nos concentramos en  $\mathrm{bcd}_1^\mathcal{R}(\mathbb{SW}_{d,\tau}f)$, que le pasa a la longitud de la barra mas larga si  $\tau$ toma valores sub-optimos  (e.g., 6.0 o 0.1). Experimenta con el codigo arriba.

2. Que pasa si el nivel de ruido de la serie de tiempo aumenta? Experimenta con el codigo arriba.

**Tu respuesta:**

1. **Se desploma.** Corriendo la celda de arriba con `noise_level = 0.1` y varios $\tau$ (persistencia maxima en $\mathrm{bcd}_1$, sin cota `thresh`):

   | $\tau$ | 0.1 | 0.5 | 1.0 | $2\pi/3 \approx 2.094$ | 2.5 | 3.0 | 6.0 | 6.1 |
   |---|---|---|---|---|---|---|---|---|
   | max pers. $H_1$ | 0.08 | 0.78 | 1.57 | **1.51** | 1.12 | 0.09 | 0.32 | 0.15 |

   Con $\tau$ sub-optimo la nube se aplasta contra la diagonal, el agujero se hace diminuto y la barra larga se vuelve indistinguible de las barras de ruido — la senal *sigue siendo periodica*, pero SW1PerS ya no lo ve. Alrededor del optimo $\tau = $ `period/(d+1)` la curva es plana (por eso $\tau = 1.0$ y $\tau = 2.094$ dan practicamente lo mismo): no hay que afinar $\tau$ con precision, basta con no caer cerca de $0$ ni cerca de un multiplo del periodo.

2. **Baja, hasta que el agujero se tapa.** Con $\tau$ optimo:

   | `noise_level` | 0.0 | 0.05 | 0.1 | 0.3 | 0.5 |
   |---|---|---|---|---|---|
   | max pers. $H_1$ | 2.07 | 1.81 | 1.51 | 0.67 | 0.27 |

   El ruido engorda el anillo y le come el agujero, asi que la barra larga se acorta de forma monotona. **Cuidado con la trampa:** si se sigue subiendo el ruido (`noise_level` = 1, 2, ...) el numero *vuelve a crecer* (0.45, 0.84, ...), pero no porque la senal sea mas periodica sino porque la nube entera se hace mas grande y la persistencia cruda tiene unidades de longitud. Por eso el puntaje de SW1PerS no usa la persistencia cruda: primero se centra cada ventana y se proyecta la nube a la esfera unitaria, y solo entonces se mide. Eso es justo lo que hacemos en la Parte IV.

---

# Part IV: Sliding Windows and 1-Persistence Scoring (SW1PerS)

Esta actividad recoge todo lo aprendido hasta este momento.

Primero generamos varias series de tiempo preiodicas con varios niveles de ruido, el nivel de ruido (menor a mayor) nos da el ranking the periodicidad `true_ranking` de la series de tiempo .

In [ ]:
# Generate a list of synthetic periodic signals corrupted with varying levels of random noise
np.random.seed(2)
t_vals = np.linspace(0,2*np.pi, 1000)

noise = np.random.randn(*t_vals.shape)
noise = noise/np.max(np.abs(noise))

n_signals = 20  # number of signals to be generated

Y_vals = []

for i, noise_level in enumerate(np.linspace(0,1,n_signals)):
  y_vals = (1- noise_level)*np.sin(5*t_vals - np.pi*np.random.rand()) + noise_level*noise
  Y_vals.append(y_vals)

true_ranking = np.random.permutation(n_signals) +1

Y_vals = np.array([Y_vals[i-1] for i in true_ranking])


# Plot the first few signals
n_plots = 7
plt.figure(figsize = (5, 5))

for i in range(n_plots):
  plt.subplot(n_plots, 1, i+1)
  plt.plot(t_vals, Y_vals[i])

In [ ]:
print('El ranking de estas sen~ales de acuerdo a su periodicidad es: ', true_ranking[0:n_plots])

## Comparando Rankings:

El *tau de Kendall* es una medida de similitud entre dos rankings:

$$\mathrm{kendall(ranking_1, ranking_2)} \; \in \; [-1, 1]$$

es un numero entre  1 (el mismo ranking) y -1 (rankings inversos).  

In [ ]:
# Compare dos rankings usando el tau de Kendall:

from scipy.stats import kendalltau

random_ranking = np.arange(n_signals).tolist()

k_tau, _ =  kendalltau(true_ranking, random_ranking)

print('La similitud entre el ranking real de las series y un ranking aleatorio es ', k_tau)


**Actividad**

1. Aplique el algoritmo de SW1PerS (nube de ventanas deslizantes --> persistencia homologica --> longitud de la barra 1-dimensional mas larga) para calcular un puntaje de periodicidad para cada serie de tiempo en los datos (`Y_vals`). **Stop and think:** usaste los parametros apropiados de $\tau$ y $d$?

2. La lista de puntajes de periodicidad calculada en la primera parte de esta actividad se pueden usar para derivar un `TDA_ranking` (mayor a menor periodicidad). Cual es la similitud entre este ranking y el original (`true_ranking`)? **Stop and think:** Hay alguna innovacion que puedas aplicar para mejorar la similitud entre estos rankings? *Hint:* Ruido. **Aplicacion real:** Este mismo metodo fue usado en  [SW1PerS, Perea et. al., **BMC Bioinformatics**, 2015,](https://pubmed.ncbi.nlm.nih.gov/26277424) para encontrar genes periodicos en sistemas biologicos.

In [ ]:
### Tu codigo aqui  --  SW1PerS

from scipy.stats import rankdata

# ---------------------------------------------------------------------
# 1. Parametros: "Stop and think: usaste los parametros apropiados?"
# ---------------------------------------------------------------------
# Las senales son  (1-e)*sin(5*t - phi) + e*ruido  sobre  t in [0, 2*pi],
# o sea que dan L = 5 vueltas: su periodo es 2*pi/5, NO 2*pi.
#
# La teoria de ventanas deslizantes pide dos cosas:
#   (a) d + 1 > 2L, para que la nube SW no se auto-interseque
#       (sin(5t) tiene armonico L = 5  =>  d = 10 sirve, d = 2 NO).
#   (b) d*tau ~ un periodo, i.e.  tau = periodo/(d + 1),
#       para que la nube quede lo mas redonda posible.
L      = 5
period = 2*np.pi / L

d   = 10                # d + 1 = 11 > 2L = 10
tau = period / (d + 1)  # la ventana d*tau cubre ~ un periodo

n_points = 400          # puntos en cada nube de ventanas deslizantes


def SW1PerS(y_vals, t_vals, d, tau, n_points=400, normalize=True):
    """Puntaje de periodicidad de una serie de tiempo.

    nube de ventanas deslizantes --> persistencia --> barra 1-dim mas larga.

    Devuelve (max_pers, score) donde score = 1 - max_pers/sqrt(3) esta en
    [0,1]:  score ~ 0  <=>  muy periodica,  score ~ 1  <=>  nada periodica.
    """
    # (i) nube de ventanas deslizantes
    SW = SW_cloud(np.array([t_vals, y_vals]), tau, d, n_points)

    if normalize:
        # (ii) centrar cada ventana (quita la media / tendencia) y
        #      proyectar a la esfera unitaria (quita la amplitud).
        #      Sin esto el puntaje mide TAMANO y no FORMA: una senal muy
        #      ruidosa tiene nube grande y persistencia cruda alta.
        SW = SW - SW.mean(axis=1, keepdims=True)
        SW = SW / np.linalg.norm(SW, axis=1, keepdims=True)

    # (iii) persistencia 1-dimensional (coeficientes en Z/11 por robustez)
    dgm1 = ripser(SW, maxdim=1, coeff=11)['dgms'][1]

    max_pers = np.max(dgm1[:, 1] - dgm1[:, 0]) if len(dgm1) > 0 else 0.0
    return max_pers, 1 - max_pers/np.sqrt(3)


# ---------------------------------------------------------------------
# 2. Un puntaje por cada serie de tiempo
# ---------------------------------------------------------------------
max_pers_list = np.array([SW1PerS(y, t_vals, d, tau, n_points)[0]
                          for y in Y_vals])
scores = 1 - max_pers_list/np.sqrt(3)

# ranking TDA: 1 = mas periodica  =>  mayor persistencia  =>  menor score
TDA_ranking = rankdata(scores, method='ordinal')

k_tau, _ = kendalltau(true_ranking, TDA_ranking)

print('true_ranking :', true_ranking)
print('TDA_ranking  :', TDA_ranking)
print('\nKendall tau (SW1PerS vs. real) = %.4f' % k_tau)


# ---------------------------------------------------------------------
# 3. "Hay alguna innovacion para mejorar la similitud?"  Hint: Ruido.
# ---------------------------------------------------------------------
# Todas las senales estan contaminadas con EL MISMO ruido de alta
# frecuencia. Ese ruido es lo que engorda la nube SW y le tapa el agujero.
# Si suavizamos la serie (promedio movil) antes de calcular SW1PerS,
# quitamos las frecuencias altas sin tocar el armonico L = 5 que nos
# interesa, y el puntaje queda mucho mas limpio.

def suaviza(y, w=5):
    """Promedio movil circular de ventana w (la senal es periodica)."""
    k = np.ones(w)/w
    y_ext = np.r_[y[-w:], y, y[:w]]          # extension periodica
    return np.convolve(y_ext, k, mode='same')[w:-w]


max_pers_smooth = np.array([SW1PerS(suaviza(y, 5), t_vals, d, tau, n_points)[0]
                            for y in Y_vals])
scores_smooth = 1 - max_pers_smooth/np.sqrt(3)
TDA_ranking_smooth = rankdata(scores_smooth, method='ordinal')

k_tau_smooth, _ = kendalltau(true_ranking, TDA_ranking_smooth)
print('Kendall tau (SW1PerS + suavizado) = %.4f' % k_tau_smooth)


# ---------------------------------------------------------------------
# 4. Comparacion con parametros mal escogidos (para ver que si importan)
# ---------------------------------------------------------------------
mp_malo = np.array([SW1PerS(y, t_vals, d=2, tau=period/3,
                            n_points=n_points)[0] for y in Y_vals])
k_tau_malo, _ = kendalltau(true_ranking, rankdata(-mp_malo, method='ordinal'))

print('\n--- resumen ---')
print('ranking aleatorio                       : %+.4f' % kendalltau(true_ranking, np.arange(n_signals))[0])
print('SW1PerS con d = 2  (d+1 <= 2L: mal)     : %+.4f' % k_tau_malo)
print('SW1PerS con d = 10 (parametros teoricos): %+.4f' % k_tau)
print('SW1PerS con d = 10 + suavizado          : %+.4f' % k_tau_smooth)


# ---------------------------------------------------------------------
# 5. Grafica: puntaje vs. ranking real
# ---------------------------------------------------------------------
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.scatter(true_ranking, scores, label='SW1PerS', s=25)
plt.scatter(true_ranking, scores_smooth, label='SW1PerS + suavizado', s=25, marker='x')
plt.xlabel('ranking real (1 = mas periodica)')
plt.ylabel('puntaje  $1 - \\mathrm{pers}/\\sqrt{3}$')
plt.title('Puntaje vs. periodicidad real')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(true_ranking, TDA_ranking, label='SW1PerS ($\\tau_K$ = %.2f)' % k_tau, s=25)
plt.scatter(true_ranking, TDA_ranking_smooth, label='+ suavizado ($\\tau_K$ = %.2f)' % k_tau_smooth, s=25, marker='x')
plt.plot([1, n_signals], [1, n_signals], 'k--', lw=1, label='ranking perfecto')
plt.xlabel('ranking real')
plt.ylabel('ranking TDA')
plt.title('Ranking TDA vs. ranking real')
plt.legend()

plt.tight_layout()
plt.show()

### Conclusiones de la Parte IV

**Resultados** (con `np.random.seed(2)`, tal como esta generado el conjunto de datos):

| metodo | Kendall $\tau$ |
|---|---|
| ranking aleatorio (`np.arange`) | $+0.22$ |
| SW1PerS con $d = 2$, $\tau = \text{periodo}/3$ | $+0.34$ |
| **SW1PerS con $d = 10$, $\tau = \text{periodo}/(d{+}1)$** | $\mathbf{+0.97}$ |
| **SW1PerS con $d = 10$ + suavizado (promedio movil, $w = 5$)** | $\mathbf{+0.99}$ |

**1. Stop and think: usaste los parametros apropiados de $\tau$ y $d$?**

No los de la Parte III. Alli la senal era $\sin(t)$, con un solo periodo, y $d = 2$, $\tau = 2\pi/3$ funcionaban. Aqui las senales son $\sin(5t - \phi)$: tienen $L = 5$ armonicos y su periodo es $2\pi/5$, no $2\pi$. Hay que cambiar dos cosas:

* **El periodo.** $\tau = \text{periodo}/(d+1)$ con periodo $= 2\pi/5$; usar $2\pi$ pondria la ventana cinco veces mas larga de lo debido.
* **La dimension.** La teoria de ventanas deslizantes (Perea–Harer) pide $d + 1 > 2L$ para que la nube $\mathbb{SW}_{d,\tau}f$ sea un encaje del circulo y no se auto-interseque. Con $L = 5$ hace falta $d \geq 10$. Con $d = 2$ el resultado es desastroso ($\tau_K = +0.34$, apenas mejor que azar) y por una razon concreta: si $d = 2$, al centrar cada ventana la nube queda en un plano de $\mathbb{R}^3$ y al normalizarla cae sobre un circulo *siempre*, sea la senal periodica o no. El puntaje deja de discriminar.

Hay un tercer detalle que no es de $d$ ni de $\tau$ pero es igual de importante: **normalizar**. Como vimos en la Parte III, la persistencia cruda tiene unidades de longitud, asi que una senal muy ruidosa puede tener nube grande y persistencia alta sin ser periodica. Centrando cada ventana (le quita la media) y proyectando a la esfera unitaria (le quita la amplitud), la persistencia maxima queda acotada por $\sqrt{3}$ y el puntaje

$$\mathrm{SW1PerS}(f) \;=\; 1 - \frac{\max \mathrm{pers}\big(\mathrm{bcd}_1^\mathcal{R}(\mathbb{SW}_{d,\tau}f)\big)}{\sqrt{3}} \;\in\; [0,1]$$

es comparable entre senales: $0$ = perfectamente periodica, $1$ = nada periodica.

**2. Similitud con el ranking real, y la innovacion (*Hint: Ruido*).**

Con los parametros correctos el ranking de SW1PerS coincide casi exactamente con el real: $\tau_K = +0.97$, y los pocos desacuerdos estan todos en la cola de senales muy ruidosas (rankings 17–20), donde la senal ya casi no existe y el orden "verdadero" es en si mismo poco significativo.

La innovacion que sugiere el *hint* es atacar el ruido antes de hacer topologia. Las 20 senales estan contaminadas con **el mismo** vector de ruido blanco, que vive en frecuencias mucho mas altas que el armonico $L = 5$ que nos interesa. Un promedio movil circular de ventana $w = 5$ (o un filtro pasa-bajas, o un ajuste por minimos cuadrados a los primeros armonicos de Fourier) elimina buena parte de ese ruido sin tocar la senal: el anillo de $\mathbb{SW}_{d,\tau}f$ adelgaza, el agujero se agranda y la persistencia vuelve a separar bien las senales de la cola. El resultado sube a $\tau_K = +0.99$ (una sola transposicion de diferencia con el ranking real). Suavizados mas agresivos ($w = 11, 21, 31$) dan entre $+0.98$ y $+0.99$: el metodo es estable, no hay que afinar $w$.

Un detalle util: al suavizar, los puntajes de las senales *poco* ruidosas casi no cambian (ya estaban cerca de 0), mientras que los de las senales ruidosas bajan bastante — se ve en el panel izquierdo de la figura. Es decir, el suavizado no "hace trampa" inflando todo, sino que recupera senal periodica que el ruido estaba escondiendo.

**Aplicacion real.** Este es exactamente el pipeline de [SW1PerS, Perea et al., *BMC Bioinformatics*, 2015](https://pubmed.ncbi.nlm.nih.gov/26277424): puntuar miles de perfiles de expresion genica por periodicidad para descubrir genes bajo control circadiano o de ciclo celular, sin suponer de antemano la forma de onda (que es la limitacion de los metodos basados en Fourier o en ajustar sinusoides).